In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import re, math, textwrap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import re
import warnings
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
warnings.filterwarnings("ignore")

# Optional: nice dataframe display in notebooks (falls back to print)
try:
    from IPython.display import display  # type: ignore
    def display_df(title, x):
        print(title)
        display(x)
except Exception:
    def display_df(title, x):
        print(title)
        print(x)

# Set the input path ONCE (edit this one line only)
OPEN_FIELD_XLSX_PATH = "/Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/csv_files/open_field.xlsx"
OPEN_FIELD_DIR = os.path.dirname(OPEN_FIELD_XLSX_PATH)
OPEN_FIELD_OUTDIR = os.path.join(OPEN_FIELD_DIR, "OpenFieldAnalysis")
os.makedirs(OPEN_FIELD_OUTDIR, exist_ok=True)
print("Saving outputs to:", OPEN_FIELD_OUTDIR)

df_open_field = pd.read_excel(OPEN_FIELD_XLSX_PATH)

# check if parket file exists and is up to date, otherwise save to parquet for faster loading next time

PARQUET_PATH = os.path.join(OPEN_FIELD_OUTDIR, "open_field.parquet")
df_open_field.to_parquet(PARQUET_PATH, index=False)
df_open_field = pd.read_parquet(PARQUET_PATH)

df_open_field.head(), df_open_field.shape, df_open_field.columns.tolist()[:5]

Saving outputs to: /Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/csv_files/OpenFieldAnalysis


(                                            filename  channel  cluster  \
 0  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        1   
 1  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        2   
 2  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        3   
 3  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        4   
 4  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        5   
 
        trial_type  theta_mod_v1  theta_mod_v2  theta_mod_v3  peak_rate  \
 0  ['openfieldb']      0.064615      1.000000      0.354456   4.176617   
 1  ['openfieldb']     -0.837708      1.000000     -0.013879   0.317807   
 2  ['openfieldb']     -0.786503      0.163636     -0.009983   8.747373   
 3  ['openfieldb']     -0.143263      0.167513     -0.024436  10.967549   
 4  ['openfieldb']      0.038483      0.409091     -0.023242   5.071136   
 
    mean_firing_rate   ahp_decay  ...  phase_locking_pval  mouse_name  \
 0          0.107410  3

In [2]:
"""
Stats functions and plotting utilities for Open field analysis
"""


import itertools
from statsmodels.stats.weightstats import ttest_ind

def _find_col(df, names):
    def norm(s):
        return re.sub(r"[\s_]+", "", str(s)).lower()
    norm_map = {norm(c): c for c in df.columns}
    for n in names:
        key = norm(n)
        if key in norm_map:
            return norm_map[key]
    raise KeyError(f"Could not find any of {names}. Available columns: {list(df.columns)}")

def _sig_stars(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def _format_p(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

def _build_order(levels, preferred_tokens):
    levels = [lv for lv in levels if pd.notna(lv)]
    used = set()
    out = []

    def norm(x): 
        return str(x).strip().lower()

    for tok in preferred_tokens:
        tok_n = tok.lower()
        match = None
        for lv in levels:
            lv_n = norm(lv)
            if lv_n == tok_n or tok_n in lv_n:
                match = lv
                break
        if match is not None and match not in used:
            out.append(match)
            used.add(match)

    for lv in levels:
        if lv not in used:
            out.append(lv)
    return out

def _holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    if m == 0:
        return pvals
    order = np.argsort(pvals)
    p_sorted = pvals[order]
    adj_sorted = np.minimum(1.0, (m - np.arange(m)) * p_sorted)
    adj_sorted = np.maximum.accumulate(adj_sorted)  # enforce monotonicity
    adj = np.empty_like(adj_sorted)
    adj[order] = adj_sorted
    return adj

def _draw_bracket(ax, x1, x2, y, h, text, fontsize=8):
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.0, c="black", zorder=8)
    ax.text((x1 + x2) / 2, y + h, text, ha="center", va="bottom", fontsize=fontsize, zorder=9)

def analyze_open_field_by_celltype_and_genotype(
    df_open_field: pd.DataFrame,
    outdir: str,
    celltype_col: str | None = None,
    genotype_col: str | None = None,
    exclude_numeric_cols: list[str] | None = None,
    min_n: int = 10,
):
    if celltype_col is None:
        celltype_col = _find_col(df_open_field, ["Cell Type", "Cell type", "cell_type", "celltype"])
    if genotype_col is None:
        genotype_col = _find_col(
            df_open_field,
            ["Genotype", "Phenotype", "genotype", "geno", "genotype_group", "Genotype group"],
        )

    print("Using CELLTYPE_COL =", celltype_col)
    print("Using GENOTYPE_COL =", genotype_col)

    numeric_cols = df_open_field.select_dtypes(include=[np.number]).columns.tolist()
    exclude = set(exclude_numeric_cols or ["tetrode", "cluster", "mouse_name"])
    cols_to_use = [c for c in numeric_cols if c not in exclude]

    # ---------- 1) 2-way ANOVA: Genotype x Cell type ----------
    anova_rows = []
    term_gt = f'C(Q("{genotype_col}"))'
    term_ct = f'C(Q("{celltype_col}"))'
    term_int = f"{term_gt}:{term_ct}"

    for metric in cols_to_use:
        test_data = df_open_field[[metric, genotype_col, celltype_col]].dropna()
        if len(test_data) < min_n:
            continue
        if test_data[genotype_col].nunique() < 2 or test_data[celltype_col].nunique() < 2:
            continue

        try:
            formula = f'Q("{metric}") ~ {term_gt} * {term_ct}'
            model = ols(formula, data=test_data).fit()
            aov = anova_lm(model, typ=2)

            p_gt  = float(aov.loc[term_gt,  "PR(>F)"]) if term_gt  in aov.index else np.nan
            p_ct  = float(aov.loc[term_ct,  "PR(>F)"]) if term_ct  in aov.index else np.nan
            p_int = float(aov.loc[term_int, "PR(>F)"]) if term_int in aov.index else np.nan

            f_gt  = float(aov.loc[term_gt,  "F"]) if term_gt  in aov.index and "F" in aov.columns else np.nan
            f_ct  = float(aov.loc[term_ct,  "F"]) if term_ct  in aov.index and "F" in aov.columns else np.nan
            f_int = float(aov.loc[term_int, "F"]) if term_int in aov.index and "F" in aov.columns else np.nan

            anova_rows.append({
                "column": metric,
                "n_observations": int(len(test_data)),
                "n_genotypes": int(test_data[genotype_col].nunique()),
                "n_cell_types": int(test_data[celltype_col].nunique()),
                "F_Genotype": f_gt,
                "F_CellType": f_ct,
                "F_Interaction": f_int,
                "p_Genotype": p_gt,
                "p_CellType": p_ct,
                "p_Interaction": p_int,
                "sig_Genotype": _sig_stars(p_gt),
                "sig_CellType": _sig_stars(p_ct),
                "sig_Interaction": _sig_stars(p_int),
            })
        except Exception as e:
            print(f"Could not run ANOVA for {metric}: {e}")

    df_anova = pd.DataFrame(anova_rows).sort_values("p_Genotype", na_position="last")
    display_df("2-way ANOVA: Genotype x Cell type", df_anova)

    anova_save_path = os.path.join(outdir, "open_field_2way_anova_genotype_celltype.csv")
    df_anova.to_csv(anova_save_path, index=False)
    print(f"Saved ANOVA table to: {anova_save_path}")

    aov_lookup = df_anova.set_index("column").to_dict(orient="index")

    # ---------- 2) Boxplots PDF: metric by Cell type (hue=Genotype) ----------
    pdf_path = os.path.join(outdir, "open_field_boxplots_by_genotype_and_celltype.pdf")

    with PdfPages(pdf_path) as pdf:
        fig = plt.figure(figsize=(11.69, 8.27))
        gt_levels = sorted(df_open_field[genotype_col].dropna().unique().tolist())
        ct_levels = sorted(df_open_field[celltype_col].dropna().unique().tolist())
        txt = (
            "Open field dataset - Boxplots by Genotype and Cell type\n"
            f"Genotypes: {', '.join(map(str, gt_levels))}\n"
            f"Cell types: {', '.join(map(str, ct_levels))}\n"
            "Black diamonds = medians (value printed)\n"
            "Brackets = pairwise genotype p-values within each cell type (Holm-adjusted per cell type)\n"
        )
        fig.text(0.05, 0.9, txt, fontsize=14, va="top")
        plt.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        hue_order = _build_order(gt_levels, ["WT", "NLGF"])
        x_order = _build_order(ct_levels, ["pyramidal", "interneurons"])

        for metric in cols_to_use:
            plot_data = df_open_field[[metric, genotype_col, celltype_col]].dropna()
            if len(plot_data) == 0:
                continue
            if plot_data[celltype_col].nunique() < 2:
                continue
            if plot_data[genotype_col].nunique() < 2:
                continue

            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            # --- palette: make WT lighter blue ---
            base = sns.color_palette("tab10", n_colors=len(hue_order))
            palette = dict(zip(hue_order, base))
            
            for g in hue_order:
                if "wt" in str(g).strip().lower():
                    palette[g] = sns.color_palette("Blues", 6)[2]  # lighter blue
                if "nlgf" in str(g).strip().lower():
                    palette[g] = sns.color_palette("Oranges", 6)[2]  # lighter orange

            sns.boxplot(
                data=plot_data,
                x=celltype_col,
                y=metric,
                hue=genotype_col,
                order=x_order,
                hue_order=hue_order,
                palette=palette,
                showfliers=False,
                ax=ax,
            )
            sns.stripplot(
                data=plot_data,
                x=celltype_col,
                y=metric,
                hue=genotype_col,
                order=x_order,
                hue_order=hue_order,
                palette=palette,
                dodge=True,
                jitter=0.28,
                size=2.5,
                alpha=0.35,
                linewidth=0.25,
                ax=ax,
            )

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax.legend(
                by_label.values(),
                by_label.keys(),
                title=genotype_col,
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                borderaxespad=0,
            )

            # ---- Title + F-stats (2nd line) ----
            row = aov_lookup.get(metric, {})
            f_gt = row.get("F_Genotype", np.nan)
            f_ct = row.get("F_CellType", np.nan)
            f_int = row.get("F_Interaction", np.nan)

            p_gt  = row.get("p_Genotype", np.nan)
            p_ct  = row.get("p_CellType", np.nan)
            p_int = row.get("p_Interaction", np.nan)
            
            fline = (
                f"F_Genotype={f_gt:.3f} (p={_format_p(p_gt)})    "
                f"F_CellType={f_ct:.3f} (p={_format_p(p_ct)})    "
                f"F_Interaction={f_int:.3f} (p={_format_p(p_int)})"
            )
            ax.set_title(f"{metric} by Cell type (hue=Genotype)\n{fline}")

            ax.set_xlabel(celltype_col)
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=45)

            # ---- Median overlay (diamond + value) ----
            med = (
                plot_data
                .groupby([celltype_col, genotype_col], dropna=False)[metric]
                .median()
                .reset_index(name="median")
            )

            m = max(1, len(hue_order))
            box_width = 0.8
            y0, y1 = ax.get_ylim()
            ypad = 0.03 * (y1 - y0)
            label_pad = 0.01 * (y1 - y0)

            for i, ct in enumerate(x_order):
                for j, gt in enumerate(hue_order):
                    sub = med[(med[celltype_col] == ct) & (med[genotype_col] == gt)]
                    if sub.empty:
                        continue
                    med_val = float(sub["median"].iloc[0])
                    offset = (j - (m - 1) / 2) * (box_width / m)
                    x_pos = i + offset
                    ax.scatter([x_pos], [med_val], color="black", s=22, marker="D", zorder=6)
                    ax.text(x_pos, med_val + label_pad, f"{med_val:.3g}", ha="center", va="bottom", fontsize=7, zorder=7, fontweight="bold")

            # ---- Pairwise genotype p-values within each cell type (brackets) ----
            geno_to_index = {g: idx for idx, g in enumerate(hue_order)}
            ct_to_index = {ct: idx for idx, ct in enumerate(x_order)}

            # precompute stacking baseline per cell type
            stack_y = {}
            for ct in x_order:
                df_ct = plot_data[plot_data[celltype_col] == ct]
                if df_ct.empty:
                    continue
                stack_y[ct] = float(df_ct[metric].max()) + ypad

            for ct in x_order:
                df_ct = plot_data[plot_data[celltype_col] == ct]
                if df_ct.empty:
                    continue
                present_genos = [g for g in hue_order if g in set(df_ct[genotype_col].unique())]
                if len(present_genos) < 2:
                    continue

                pairs = []
                raw_p = []
                for g1, g2 in itertools.combinations(present_genos, 2):
                    x1 = df_ct[df_ct[genotype_col] == g1][metric].dropna().to_numpy()
                    x2 = df_ct[df_ct[genotype_col] == g2][metric].dropna().to_numpy()
                    if len(x1) < 2 or len(x2) < 2:
                        continue
                    _, p, _ = ttest_ind(x1, x2, usevar="unequal")
                    pairs.append((g1, g2))
                    raw_p.append(float(p))

                if not raw_p:
                    continue

                p_adj = _holm_adjust(raw_p)

                i = ct_to_index[ct]
                for (g1, g2), padj in zip(pairs, p_adj):
                    j1 = geno_to_index[g1]
                    j2 = geno_to_index[g2]
                    off1 = (j1 - (m - 1) / 2) * (box_width / m)
                    off2 = (j2 - (m - 1) / 2) * (box_width / m)
                    x_left = i + off1
                    x_right = i + off2
                    if x_left > x_right:
                        x_left, x_right = x_right, x_left

                    y = stack_y.get(ct, float(df_ct[metric].max()) + ypad)
                    h = 0.01 * (y1 - y0)
                    label = f"p={_format_p(padj)} ({_sig_stars(padj)})"
                    _draw_bracket(ax, x_left, x_right, y, h, label, fontsize=8)
                    stack_y[ct] = y + ypad  # bump for next bracket

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    print(f"Saved boxplots PDF to: {pdf_path}")
    return df_anova, anova_save_path, pdf_path


# Run it
df_anova_genotype_celltype, anova_csv, boxplot_pdf = analyze_open_field_by_celltype_and_genotype(
    df_open_field=df_open_field,
    outdir=OPEN_FIELD_OUTDIR,
)

Using CELLTYPE_COL = Cell type
Using GENOTYPE_COL = Phenotype
2-way ANOVA: Genotype x Cell type


,column,n_observations,n_genotypes,n_cell_types,F_Genotype,F_CellType,F_Interaction,p_Genotype,p_CellType,p_Interaction,sig_Genotype,sig_CellType,sig_Interaction
12,half_split_stability,2186,2,2,381.421660,233.722905,35.390821,2.055389e-78,3.341351e-50,3.133867e-09,***,***,***
11,odd_even_stability,2214,2,2,349.584891,217.533542,17.588833,1.537779e-72,4.982128e-47,2.850126e-05,***,***,***
18,coherence,2214,2,2,258.643767,250.463474,32.609487,4.025453e-55,1.599112e-53,1.278375e-08,***,***,***
22,Age_weeks,2214,2,2,208.595250,8.253314,0.223957,2.991573e-45,4.106576e-03,6.360881e-01,***,**,ns
4,peak_rate,2214,2,2,58.475927,114.485734,0.729534,3.048913e-14,4.385572e-26,3.931270e-01,***,***,ns
17,spatial_info,2214,2,2,54.684807,204.911200,6.151674,1.997764e-13,1.625341e-44,1.320274e-02,***,***,*
2,theta_mod_v2,2095,2,2,36.783155,202.079761,1.306853,1.562587e-09,7.567394e-44,2.530961e-01,***,***,ns
3,theta_mod_v3,2214,2,2,31.871363,42.603416,0.044433,1.858945e-08,8.288844e-11,8.330691e-01,***,***,ns
7,theta_modulation,2214,2,2,31.871363,42.603416,0.044433,1.858945e-08,8.288844e-11,8.330691e-01,***,***,ns
8,field_size,1080,2,2,29.752739,36.549858,0.903754,6.087927e-08,2.049399e-09,3.419907e-01,***,***,ns


Saved ANOVA table to: /Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/csv_files/OpenFieldAnalysis/open_field_2way_anova_genotype_celltype.csv
Saved boxplots PDF to: /Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/csv_files/OpenFieldAnalysis/open_field_boxplots_by_genotype_and_celltype.pdf
